In [ ]:
import os
import sys
import seaborn as sns
import pandas as pd
import numpy as np

repository_path = "/Users/user/Downloads/sift2_unbiased/"
functions_path  = os.path.join(repository_path, "code", "paper_figures", "functions")
sys.path.append(os.path.abspath(functions_path))

from update_index import update_index
from streamline_count import streamline_count
from extract_bundles import extract_bundles
from compute_groundtruth_bundles import compute_groundtruth_bundles
from sum_upper_triangle import sum_upper_triangle

from load_streamline_count import load_streamline_count
from load_sift2_cross import load_sift2_cross
from load_sift2_temp import load_sift2_temp
from load_sift2_sym import load_sift2_sym
from load_sift2_diff import load_sift2_diff

from plot_bundle_errors import plot_bundle_errors


In [ ]:
# Number of categories to plot
N_CATS = 4  

# Build custom 4‐color palette :
GLOBAL_PALETTE = [
    "red",         
    "orange",  
    "steelblue",   
    "lightblue"    
]

order = ["Pipeline 1", "Pipeline 2", "Pipeline 3", "Pipeline 4"]

# Register it as Seaborn’s default
sns.set_palette(GLOBAL_PALETTE)

In [ ]:
# =============================
# PARAMETERS & PATHS
# =============================

# Choose phantom, tractography string, and ground-truth file identifiers
phantom = "disco3_central_lesion_snr20"   # options: disco3_crossing_bundles_snr20, disco3_single_bundle_snr20, disco3_central_lesion_snr20 
phantom_path = os.path.join(repository_path,"data","phantoms",phantom)
tcks_cross = "tracks"  
tcks_unbiased = "tracks_template"  

# SIFT2 Cross-sectional (formerly SIFT2 Absolute)
reg_basis_abs = "streamline"
reg_fn_abs = "gamma"
reg_strength_abs = 0.1

# SIFT2 Symmetric
reg_basis_sym = "streamline"
reg_strength_sym = 0.1

# SIFT2 Differential
reg_fn_diff = "dualinvbarr"
reg_basis_diff = "streamline"
reg_strength_diff = 0.1

In [ ]:
# Extract the true fibre count for each timepoint
fiber_count_tp1 = streamline_count(os.path.join(phantom_path,'orig/ground_truth/tp1/tracks_gt_tp1.tck'))
fiber_count_tp2 = streamline_count(os.path.join(phantom_path,'orig/ground_truth/tp2/tracks_gt_tp2.tck'))
fiber_count_tpav = (fiber_count_tp1+fiber_count_tp2)/2

In [ ]:
# =============================
# LOAD DATA & COMPUTE DIFFERENCES
# =============================

# Load ground truth connectomes and compute their difference.
gt_tp1 = update_index(pd.read_csv(f'{phantom_path}/orig/ground_truth/tp1/gt_sift2_tp1.csv', header=None))
gt_tp2 = update_index(pd.read_csv(f'{phantom_path}/orig/ground_truth/tp2/gt_sift2_tp2.csv', header=None))
gt_tp_diff = (gt_tp2 - gt_tp1).fillna(0)
gt_tp_av = (gt_tp1 + gt_tp2) / 2

# Filtering two spurious streamlines in the original phantom (self assigned to node)
gt_tp1[gt_tp1 < 3] = 0
gt_tp2[gt_tp2 < 3] = 0

# Load the pipelines.
tp1_nos, tp2_nos = load_streamline_count(phantom_path, tcks_cross,normalise=fiber_count_tp1)
tp1_cross, tp2_cross = load_sift2_cross(phantom_path, tcks_cross, reg_basis_abs, reg_fn_abs, reg_strength_abs, normalise=fiber_count_tp1)
tp1_sym, tp2_sym     = load_sift2_sym(phantom_path, tcks_unbiased, reg_basis_sym, reg_fn_abs, reg_strength_sym,normalise=fiber_count_tp1)
tp1_diff, tp2_diff   = load_sift2_diff(phantom_path, tcks_unbiased, reg_basis_abs, reg_fn_abs, reg_strength_abs, reg_fn_diff, reg_basis_diff, reg_strength_diff, normalise=fiber_count_tp1)
tp_av = load_sift2_temp(phantom_path, tcks_unbiased, reg_basis_abs, reg_fn_abs, reg_strength_abs, normalise=fiber_count_tpav)

# Compute differences (absolute difference between timepoints).
tp_diff_nos = tp2_nos - tp1_nos
tp_diff_cross = tp2_cross - tp1_cross
tp_diff_sym   = tp2_sym - tp1_sym
tp_diff_diff = tp2_diff - tp1_diff

# Compute differential errors 
error_nos = abs(tp_diff_nos - gt_tp_diff)
error_cross = abs(tp_diff_cross - gt_tp_diff)
error_sym   = abs(tp_diff_sym - gt_tp_diff)
error_diff  = abs(tp_diff_diff - gt_tp_diff)

In [ ]:
# =============================
# BUNDLE DEFINITIONS VIA GROUND TRUTH
# =============================

# Determine ground truth bundles by comparing gt_tp1 and gt_tp2.
true_bundles_with_effect, true_bundles_no_effect, false_bundles = compute_groundtruth_bundles(gt_tp1, gt_tp2)

# Combine all true positive bundles (with and without effect) and calculate mean ground truth.
true_bundles_all = true_bundles_with_effect + true_bundles_no_effect
gt_true_all = extract_bundles(gt_tp1, true_bundles_all)
mean_true_all = gt_true_all.replace(0, np.nan).stack().mean()

In [ ]:
# =============================
# EXTRACT ERROR BUNDLES FOR EACH PIPELINE
# =============================

# For Streamline Count Cross-Sectional.
error_nos_true_no_effect  = extract_bundles(error_nos, true_bundles_no_effect)
error_nos_true_with_effect = extract_bundles(error_nos, true_bundles_with_effect)
error_nos_false           = extract_bundles(error_nos, false_bundles)

# For SIFT2 Cross-Sectional.
error_cross_true_no_effect  = extract_bundles(error_cross, true_bundles_no_effect)
error_cross_true_with_effect = extract_bundles(error_cross, true_bundles_with_effect)
error_cross_false           = extract_bundles(error_cross, false_bundles)

# For SIFT2 Unbiased.
error_sym_true_no_effect    = extract_bundles(error_sym, true_bundles_no_effect)
error_sym_true_with_effect  = extract_bundles(error_sym, true_bundles_with_effect)
error_sym_false             = extract_bundles(error_sym, false_bundles)

# For SIFT2 Differential.
error_diff_true_no_effect   = extract_bundles(error_diff, true_bundles_no_effect)
error_diff_true_with_effect = extract_bundles(error_diff, true_bundles_with_effect)
error_diff_false            = extract_bundles(error_diff, false_bundles)

### PLOT FOR TRUE BUNDLES WITH EFFECT

In [ ]:
titles_list = [
    r"Pipeline 1",
    r"Pipeline 2",
    r"Pipeline 3",
    r"Pipeline 4"
]

fibre_count_tp1_true_with_effect = sum_upper_triangle(extract_bundles(gt_tp1, true_bundles_with_effect))

total_error_nos_true_with_effect = sum_upper_triangle(error_nos_true_with_effect)
total_error_cross_true_with_effect = sum_upper_triangle(error_cross_true_with_effect)
total_error_sym_true_with_effect   = sum_upper_triangle(error_sym_true_with_effect)
total_error_diff_true_with_effect  = sum_upper_triangle(error_diff_true_with_effect)

# Plot 1: Error Distribution for True Bundles with Effect.
title_main = "% Fiber Count Error"
errors_all = [error_nos_true_with_effect, error_cross_true_with_effect, error_sym_true_with_effect, error_diff_true_with_effect]
df = plot_bundle_errors(errors_all, title_main, titles_list, minimal=True, log_y=True)

print("total errors with effect:")
print(total_error_nos_true_with_effect, total_error_cross_true_with_effect, total_error_sym_true_with_effect, total_error_diff_true_with_effect)

### PLOT FOR TRUE BUNDLES WTHOUT EFFECT

In [ ]:
titles_list = [
    r"Pipeline 1",
    r"Pipeline 2",
    r"Pipeline 3",
    r"Pipeline 4"
]

fibre_count_tp1_true_no_effect = sum_upper_triangle(extract_bundles(gt_tp1, true_bundles_no_effect))

total_error_nos_true_no_effect = sum_upper_triangle(error_nos_true_no_effect)
total_error_cross_true_no_effect = sum_upper_triangle(error_cross_true_no_effect)
total_error_sym_true_no_effect   = sum_upper_triangle(error_sym_true_no_effect)
total_error_diff_true_no_effect  = sum_upper_triangle(error_diff_true_no_effect)

# Plot 1: Error Distribution for True Bundles with Effect.
title_main = "% Fiber Count Error"
errors_all = [error_nos_true_no_effect, error_cross_true_no_effect, error_sym_true_no_effect, error_diff_true_no_effect]
df = plot_bundle_errors(errors_all, title_main, titles_list, minimal=True, log_y=True)

print("total errors no effect:")
print(total_error_nos_true_no_effect, total_error_cross_true_no_effect, total_error_sym_true_no_effect, total_error_diff_true_no_effect)

### PLOT FOR FALSE POSITIVE BUNDLES

In [ ]:
titles_list = [
    r"Pipeline 1",
    r"Pipeline 2",
    r"Pipeline 3",
    r"Pipeline 4"
]

fibre_count_tp1_false = sum_upper_triangle(extract_bundles(gt_tp1, true_bundles_no_effect))

# Plot 1: Error Distribution for True Bundles with Effect.
title_main = "% Fiber Count Error"
errors_all = [error_nos_false, error_cross_false, error_sym_false, error_diff_false]
df = plot_bundle_errors(errors_all, title_main, titles_list, minimal=True, log_y=True)